### Tranform Customer Data
- Remove records with NULL Customer_id
- Remove exact duplicate records
- Remove duplicate records based on created_timestamp
- CAST the column to the correct data type
- Write transformed data to silver schema

###1. Remove records with NULL Customer_id

In [0]:
select *
from gizmobox_sivan.bronze.v_customers
where customer_id is not null

###2. Remove exact duplicate records

In [0]:
select distinct *
from gizmobox_sivan.bronze.v_customers
where customer_id is not null

###3. Remove duplicate records based on created_timestamp

In [0]:
CREATE OR REPLACE TEMPORARY VIEW tv_customers_distinct
AS
select distinct *
from gizmobox_sivan.bronze.v_customers
where customer_id is not null

In [0]:
select * from tv_customers_distinct

In [0]:
select customer_id, 
    max(created_timestamp),
    max(customer_name),
    max(date_of_birth),
    max(email),
    max(member_since),
    max(telephone)    
from tv_customers_distinct
group by customer_id;

In [0]:
WITH customers_max
AS
(
    SELECT 
        customer_id,
        max(created_timestamp) max_timestamp
    from tv_customers_distinct
    group by customer_id
)
SELECT 
    dc.*
FROM tv_customers_distinct dc
INNER JOIN customers_max cm
on dc.customer_id = cm.customer_id
and dc.created_timestamp = cm.max_timestamp


###4. CAST the column to the correct data type

In [0]:
WITH customers_max
AS
(
    SELECT 
        customer_id,
        max(created_timestamp) max_timestamp
    from tv_customers_distinct
    group by customer_id
)
SELECT 
    CAST(dc.created_timestamp as timestamp) as created_timestamp,
    dc.customer_id,
    dc.customer_name,
    CAST(dc.date_of_birth as date) as date_of_birth,
    dc.email,
    CAST(dc.member_since as date) as member_since,
    dc.telephone,
    dc.file_path
FROM tv_customers_distinct dc
INNER JOIN customers_max cm
on dc.customer_id = cm.customer_id
and dc.created_timestamp = cm.max_timestamp

### 5. Write data to a Delta table

In [0]:
CREATE TABLE gizmobox_sivan.silver.customers
AS
WITH customers_max
AS
(
    SELECT 
        customer_id,
        max(created_timestamp) max_timestamp
    from tv_customers_distinct
    group by customer_id
)
SELECT 
    CAST(dc.created_timestamp as timestamp) as created_timestamp,
    dc.customer_id,
    dc.customer_name,
    CAST(dc.date_of_birth as date) as date_of_birth,
    dc.email,
    CAST(dc.member_since as date) as member_since,
    dc.telephone,
    dc.file_path
FROM tv_customers_distinct dc
INNER JOIN customers_max cm
on dc.customer_id = cm.customer_id
and dc.created_timestamp = cm.max_timestamp;

In [0]:
select * from gizmobox_sivan.silver.customers

In [0]:
describe extended gizmobox_sivan.silver.customers;